In [32]:
import numpy as np

## 1a

At expiry $T_2$, the portfolio pays
$$(S_{T_2} - K) - (S_{T_2} - F_t) = F_t - K \quad \text{(deterministic).}$$

Discounting this deterministic payoff:

$$\boxed{\;f_t = e^{-r(T_2 - t)}(F_t - K).\;}$$

## 1b

The argument says "borrow $S_t$ dollars, buy the stock, and short the forward." For crude oil, "buy the stock" means take physical delivery of barrels and hold them, which incurs storage costs. So you accumulate more debt than $S_t e^{r(T_2-t)}$, and the cash-and-carry arbitrage breaks down.

In [33]:
# Exponential Ornstein-Uhlenbeck process

class XOU:

    def __init__(self, kappa, alpha, sigma, S0, r):

        self.kappa = kappa
        self.alpha = alpha
        self.sigma = sigma
        self.S0 = S0
        self.r = r

In [34]:
hw5dynamics=XOU(kappa = 0.472, alpha = 4.4, sigma = 0.368, S0 = 106.9, r = 0.05)

In [35]:
class CallOnForwardPrice:

    def __init__(self, K1, T1, T2):

        self.K1 = K1
        self.T1 = T1
        self.T2 = T2


In [36]:
hw5contract = CallOnForwardPrice(K1 = 103.2, T1 = 0.5, T2 = 0.75)

In [37]:
class MCengine:

    def __init__(self, N, M, epsilon, seed):

        self.N = N   # Number of timesteps on each path
        self.M = M   # Number of paths
        self.epsilon = epsilon  # For the dC/dS calculation
        self.rng = np.random.default_rng(seed=seed)

    def price_call_XOU(self, contract, dynamics):
        # Simulates X = log(S) with Euler scheme on [0, T1], then uses the closed-form
        # forward formula at T1 to compute F_{T1}, then prices the call (F_{T1} - K1)^+.
        # Reuses the same Z's for the C(S0+epsilon) bump, for variance reduction in delta.

        N, M = self.N, self.M
        kappa = dynamics.kappa
        alpha = dynamics.alpha
        sigma = dynamics.sigma
        r     = dynamics.r
        K1    = contract.K1
        T1    = contract.T1
        T2    = contract.T2

        deltat = T1 / N
        sqrt_dt = np.sqrt(deltat)

        # Pre-generate random normals (M paths x N timesteps), used for BOTH C(S0) and C(S0+eps)
        Z = self.rng.normal(size=(M, N))

        tau = T2 - T1
        a   = np.exp(-kappa * tau)
        b   = (1 - a) * alpha
        c   = (sigma**2) / (4 * kappa) * (1 - np.exp(-2 * kappa * tau))

        def discounted_payoffs(S0):
            X = np.full(M, np.log(S0))
            for n in range(N):
                X = X + kappa * (alpha - X) * deltat + sigma * sqrt_dt * Z[:, n]
            # F_{T1} from closed-form expectation of S_{T2} = exp(X_{T2}) given X_{T1}
            F_T1 = np.exp(a * X + b + c)
            payoff = np.maximum(F_T1 - K1, 0.0)
            return np.exp(-r * T1) * payoff

        payoffs       = discounted_payoffs(dynamics.S0)
        payoffs_bumped = discounted_payoffs(dynamics.S0 + self.epsilon)

        call_price     = float(np.mean(payoffs))
        standard_error = float(np.std(payoffs, ddof=1) / np.sqrt(M))
        call_delta     = float((np.mean(payoffs_bumped) - np.mean(payoffs)) / self.epsilon)

        return (call_price, standard_error, call_delta)

In [38]:
hw5MC = MCengine(N=100, M=200000, epsilon=0.01, seed=0)
# M = 200,000 keeps SE < 0.05 (call payoff std is roughly ~10, so SE ~ 10/sqrt(2e5) ≈ 0.022)

In [39]:
(call_price, standard_error, call_delta) = hw5MC.price_call_XOU(hw5contract,hw5dynamics)

In [40]:
print(call_price, standard_error, call_delta)

7.788026496905533 0.029834123248165884 0.3415465981566612


## 1c

Simulate $X_t = \log S_t$ with Euler steps on $[0, T_1]$ using the OU dynamics
$$X_{t+\Delta t} = X_t + \kappa(\alpha - X_t)\Delta t + \sigma\sqrt{\Delta t}\,Z_n,\quad Z_n \sim N(0,1).$$

At $T_1$, compute $F_{T_1}$ via the closed-form formula given in the PDF, take the discounted payoff $e^{-rT_1}(F_{T_1} - K_1)^+$, and average over $M$ paths.

Result printed above. With $M = 200{,}000$ the standard error is well below $0.05$.

## 1d

Re-runs the simulation starting at $S_0 + \varepsilon$ with $\varepsilon = 0.01$, **reusing the same $Z$ matrix**. The estimate is
$$\frac{C(S_0 + 0.01) - C(S_0)}{0.01}.$$

Reusing the same randoms is critical: the two MC estimates are highly positively correlated, so their difference has a much smaller variance than two independent MC estimates would have.

## 1e

From part (a), $f_0 = e^{-rT_2}(F_0 - K)$, where the closed-form expression for $F_0$ is
$$F_0 = \exp\!\Big[e^{-\kappa T_2}\log S_0 + (1 - e^{-\kappa T_2})\alpha + \tfrac{\sigma^2}{4\kappa}(1 - e^{-2\kappa T_2})\Big].$$

Since $K$ is a constant, $\partial f_0/\partial S = e^{-rT_2}\,\partial F_0/\partial S$. By the chain rule (only the $\log S_0$ term depends on $S_0$):

$$\frac{\partial F_0}{\partial S_0} = F_0 \cdot \frac{e^{-\kappa T_2}}{S_0}.$$

Therefore

$$\boxed{\;\frac{\partial f_0}{\partial S} = e^{-(r+\kappa)T_2}\,\frac{F_0}{S_0}.\;}$$

## 1f

To hedge a short call by going long forwards, we want the portfolio's delta with respect to spot $S$ to match the call's delta. If we hold $\theta$ forward contracts at delivery price $K$ (any $K$):

$$\theta \cdot \frac{\partial f_0}{\partial S} = \frac{\partial C}{\partial S}.$$

Therefore

$$\boxed{\;\theta = \frac{\partial C/\partial S}{\partial f_0/\partial S} = \frac{\text{call\_delta}}{e^{-(r+\kappa)T_2}\,F_0/S_0}.\;}$$

## 1g

Holder receives $\theta$ barrels at $T_2$, pays $K\theta$ at $T_2$. Choice of $\theta \in [4000, 5000]$ made at $T_1$.

Time-$T_1$ value per barrel of the deferred purchase = $e^{-r(T_2-T_1)}(F_{T_1} - K)$ (from part (a) with the role of "spot" played by the forward at $T_1$).

The holder optimally picks
$$\theta^*(F_{T_1}) = \begin{cases} 5000 & F_{T_1} > K, \\ 4000 & F_{T_1} \le K, \end{cases}$$
so the time-$T_1$ contract value is
$$V_{T_1} = e^{-r(T_2-T_1)}\big[\,4000(F_{T_1}-K) + 1000(F_{T_1}-K)^+\,\big].$$

The time-0 value is the discounted Q-expectation. Use two facts:
1. $F_t = \mathbb{E}^Q_t[S_{T_2}]$ is a $Q$-martingale (tower), so $\mathbb{E}^Q[F_{T_1}] = F_0$.
2. $C(S_0) = e^{-rT_1}\mathbb{E}^Q[(F_{T_1}-K)^+]$, the call price from part (c).

Therefore

$$\boxed{\;V_0 = 4000\,e^{-rT_2}(F_0 - K) + 1000\,e^{-r(T_2-T_1)}\,C(S_0).\;}$$

In [41]:
# Numerical values for parts (e), (f), (g)

kappa = hw5dynamics.kappa
alpha = hw5dynamics.alpha
sigma = hw5dynamics.sigma
S0    = hw5dynamics.S0
r     = hw5dynamics.r
T1    = hw5contract.T1
T2    = hw5contract.T2
K     = hw5contract.K1

# Closed-form F_0 = E[S_{T2}]
F0 = np.exp(np.exp(-kappa*T2)*np.log(S0)
            + (1 - np.exp(-kappa*T2))*alpha
            + (sigma**2)/(4*kappa)*(1 - np.exp(-2*kappa*T2)))

# (e) df0/dS
df0_dS = np.exp(-(r + kappa)*T2) * F0 / S0

# (f) hedge ratio
hedge_ratio = call_delta / df0_dS

# (g) purchase agreement value
V0_purchase = 4000 * np.exp(-r*T2) * (F0 - K) + 1000 * np.exp(-r*(T2 - T1)) * call_price

print(f"F_0                      = {F0:.4f}")
print(f"e) df_0/dS              = {df0_dS:.6f}")
print(f"f) hedge ratio (forwards)= {hedge_ratio:.4f}")
print(f"g) purchase agreement V0 = {V0_purchase:.2f}")

F_0                      = 102.2304
e) df_0/dS              = 0.646511
f) hedge ratio (forwards)= 0.5283
g) purchase agreement V0 = 3955.44


# Problem 2

## 2a

The dynamics $dS_t = rS_t\,dt + \sigma(t)S_t\,dW_t$ have $\sigma$ depending on $t$ only (not on $S$ or any other random source). Setting $X = \log S$:
$$X_T - X_0 = \int_0^T (r - \tfrac{1}{2}\sigma^2(s))\,ds + \int_0^T \sigma(s)\,dW_s,$$
so $X_T$ is normal with variance $V(T) := \int_0^T \sigma^2(s)\,ds$. This is the **same distribution** as constant-vol GBM with effective vol $\bar\sigma(T) := \sqrt{V(T)/T}$.

Therefore for **any** strike $K$:
$$C^{model}(K, T) = C^{BS}(S_0, K, r, T, \bar\sigma(T)),$$
so the implied vol is $\sigma_{imp}(K, T) = \bar\sigma(T)$, **flat in $K$**.

- **Term structure (varying with $T$):** YES — $\bar\sigma(T)$ depends on $T$ in general.
- **Skew (varying with $K$):** NO — at any fixed $T$, the implied vol is constant in $K$.

## 2b

Denote the implied vols of the 0.1-, 0.2-, 0.5-year ATM calls by $\sigma_1, \sigma_2, \sigma_3$. These are obtained by inverting Black–Scholes (computed below).

For a step function $\sigma(t) = c_1$ on $[0, 0.1]$, $c_2$ on $[0.1, 0.2]$, $c_3$ on $[0.2, 0.5]$, the consistency condition $V(T) = \sigma_{imp}^2(T) \cdot T$ gives:

$$c_1^2 \cdot 0.1 = \sigma_1^2 \cdot 0.1 \;\Rightarrow\; c_1 = \sigma_1$$
$$c_1^2 \cdot 0.1 + c_2^2 \cdot 0.1 = \sigma_2^2 \cdot 0.2 \;\Rightarrow\; c_2^2 = 2\sigma_2^2 - \sigma_1^2$$
$$c_2^2 \cdot 0.1 + c_3^2 \cdot 0.3 = \sigma_3^2 \cdot 0.5 - \sigma_1^2 \cdot 0.1 \;\Rightarrow\; c_3^2 = \tfrac{0.5\sigma_3^2 - 0.2\sigma_2^2}{0.3}$$

Numerical values computed below.

## 2c

Using the calibrated $\sigma$:

**Total variance to $T = 0.4$:**
$$V(0.4) = c_1^2 \cdot 0.1 + c_2^2 \cdot 0.1 + c_3^2 \cdot 0.2.$$

**Implied vol** $\sigma_{imp}(0.4) = \sqrt{V(0.4)/0.4}$. Plug into BS to get the price.

**Time-0.1 implied vol of the same option:**
$$\sigma_{imp}(0.1, 0.4) = \sqrt{\frac{V(0.4) - V(0.1)}{0.4 - 0.1}} = \sqrt{\frac{c_2^2 \cdot 0.1 + c_3^2 \cdot 0.2}{0.3}}.$$

Numerical values computed below.

In [43]:
from scipy.stats import norm
from scipy.optimize import brentq

# Problem 2(b)+(c) numerical computations

S0_p2 = 100.0
r_p2  = 0.05
K_p2  = 100.0   # ATM

# Given call prices at three expiries
prices = {0.1: 5.25, 0.2: 7.25, 0.5: 9.5}

def bs_call(S0, K, r, T, sigma):
    F = S0 * np.exp(r*T)
    sT = sigma*np.sqrt(T)
    d1 = (np.log(F/K) + 0.5*sigma**2*T) / sT
    d2 = d1 - sT
    return np.exp(-r*T) * (F*norm.cdf(d1) - K*norm.cdf(d2))

def implied_vol(price, S0, K, r, T):
    return brentq(lambda s: bs_call(S0, K, r, T, s) - price, 1e-6, 5.0)

# Implied vols at the three expiries
sig1 = implied_vol(prices[0.1], S0_p2, K_p2, r_p2, 0.1)
sig2 = implied_vol(prices[0.2], S0_p2, K_p2, r_p2, 0.2)
sig3 = implied_vol(prices[0.5], S0_p2, K_p2, r_p2, 0.5)
print("Implied vols:")
print(f"  sigma_imp(0.1) = {sig1:.6f}")
print(f"  sigma_imp(0.2) = {sig2:.6f}")
print(f"  sigma_imp(0.5) = {sig3:.6f}")

# Local-vol step function
c1_sq = sig1**2
c2_sq = 2*sig2**2 - sig1**2
c3_sq = (0.5*sig3**2 - 0.2*sig2**2) / 0.3
c1, c2, c3 = np.sqrt(c1_sq), np.sqrt(c2_sq), np.sqrt(c3_sq)
print("\nCalibrated step-function local vol:")
print(f"  sigma(t) = {c1:.6f} on [0.0, 0.1]")
print(f"  sigma(t) = {c2:.6f} on [0.1, 0.2]")
print(f"  sigma(t) = {c3:.6f} on [0.2, 0.5]")

# 2(c): T=0.4 ATM call
V_04 = c1_sq*0.1 + c2_sq*0.1 + c3_sq*0.2
sig_imp_04 = np.sqrt(V_04 / 0.4)
price_04   = bs_call(S0_p2, K_p2, r_p2, 0.4, sig_imp_04)

# Time-0.1 implied vol of the T=0.4 call
V_01_04 = c2_sq*0.1 + c3_sq*0.2     # variance contribution from t=0.1 to t=0.4
sig_imp_01_04 = np.sqrt(V_01_04 / 0.3)

print("\n2c) results:")
print(f"  Total variance V(0.4)        = {V_04:.6f}")
print(f"  sigma_imp(0.4)               = {sig_imp_04:.6f}")
print(f"  Time-0 price of T=0.4 call   = {price_04:.4f}")
print(f"  Time-0.1 implied vol         = {sig_imp_01_04:.6f}")

Implied vols:
  sigma_imp(0.1) = 0.397320
  sigma_imp(0.2) = 0.380171
  sigma_imp(0.5) = 0.295097

Calibrated step-function local vol:
  sigma(t) = 0.397320 on [0.0, 0.1]
  sigma(t) = 0.362211 on [0.1, 0.2]
  sigma(t) = 0.220871 on [0.2, 0.5]

2c) results:
  Total variance V(0.4)        = 0.038663
  sigma_imp(0.4)               = 0.310897
  Time-0 price of T=0.4 call   = 8.7842
  Time-0.1 implied vol         = 0.276143
